## The `pg_step_size` sweep — run this INSTEAD of cell 7, not after it

The 19 Sep n=10 run answered the only question that mattered: at `pg_step_size=1` the shield
engages on **0–1 of 10** images, and the one is inside the error bars. So there is nothing to wash
yet, and the four-arm table would measure generator variation.

This cell fixes the **input**. It re-runs the protect stage and the N arm at each `pg_step_size`
and reports how many images clear their floor. It is the same harness — no new method, no new
metric.

**Success = a drop in `ssim_adv` of ≥ 0.067, i.e. ≥ 5 of 10 engaged.** Below 0.030 the setting is
indistinguishable from `pg_step_size=1`. **Stop at the first setting that clears and do not sweep
past it** — the point is a working denominator, not the strongest possible shield.


In [ ]:
# 7s · SWEEP pg_step_size. Protect + N arm only, one archive tag per setting.
assert not SMOKE, 'SMOKE is still True in cell 4'
assert GPU_OK

# _ssim_pair and _pd are defined in cell 8b, which runs after this one. Define them here so the
# sweep is self-contained and can be run straight after cell 7.
from skimage.metrics import structural_similarity as _ssim
import pandas as _pd
from PIL import Image as _Im
def _ssim_pair(pa, pb):
    a = np.asarray(_Im.open(pa).convert('RGB'), np.uint8)
    b = np.asarray(_Im.open(pb).convert('RGB'), np.uint8)
    if a.shape != b.shape:
        h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
        a = np.asarray(_Im.fromarray(a).resize((w, h)))
        b = np.asarray(_Im.fromarray(b).resize((w, h)))
    return float(_ssim(a, b, channel_axis=2, data_range=255))

SWEEP      = [2, 4, 8]      # 1 is already measured: 0-1 of 10 engaged
TARGET_ENG = 5              # >= 5 of 10 clears; see RESULT-0919-n10-gate.md
BASELINE   = {'1233476865_1.png':0.5018,'1525918600_1.png':0.6372,'178046512_1.png':0.5370,
              '181707205_1.png':0.6453,'1961032923_1.png':0.6966,'2061993362_1.png':0.4656,
              '2099073485_1.png':0.5401,'2139626906_1.png':0.3837,'2167874246_1.png':0.4346,
              '221629697_1.png':0.5178}          # pg_step_size = 1, run_0919_0406

SWEEP_ROWS, _cleared = [], None
for _s in SWEEP:
    P['pg_step_size'] = _s
    # PG is an f-string built ONCE in cell 6 off P — it does NOT follow P. Rebuild it, or every
    # setting in this loop silently runs at pg_step_size=1 and the sweep reports a flat line.
    PG = (f"--attack_type={P['attack_type']} --pg_eps={P['pg_eps']} --pg_step_size={P['pg_step_size']} "
          f"--pg_eta={P['pg_eta']} --parallel_index=-1 --device={DEVICE}")

    arm = dict(name=f'N_step{_s}', pur_iters=None, masked=False)
    print(f'\n{"#"*70}\n#  pg_step_size = {_s}\n{"#"*70}', flush=True)
    try:
        run_arm(arm)
    except Exception as e:
        print(f'!! step {_s} failed: {type(e).__name__}: {e}')
        if DEVICE.startswith('cuda'): torch.cuda.empty_cache()
        continue

    d, n_eng = ARCHIVE / arm['name'], 0
    for n in sorted(os.listdir(d / 'edit_clean')):
        try:
            a = _ssim_pair(d / 'edit_protected' / n, d / 'edit_clean' / n)
        except FileNotFoundError:
            continue
        fl = SEED_FLOOR.get(n, float('nan'))
        SWEEP_ROWS.append(dict(pg_step_size=_s, image=n, ssim_adv=round(a, 4), floor=fl,
                               margin=round(a - fl, 4), engaged=bool(a < fl),
                               drop_vs_step1=round(BASELINE.get(n, float('nan')) - a, 4)))
        n_eng += int(a < fl)
    print(f'\n>>> pg_step_size={_s}: {n_eng}/10 engaged', flush=True)
    if n_eng >= TARGET_ENG:
        _cleared = _s
        print(f'>>> CLEARS at {TARGET_ENG}+. STOP HERE — do not sweep further.'); break

sweep = _pd.DataFrame(SWEEP_ROWS)
display(sweep)
if not sweep.empty:
    print()
    for _s, g in sweep.groupby('pg_step_size'):
        print(f'   pg_step_size={_s}:  {int(g.engaged.sum())}/10 engaged · '
              f'median drop vs step1 {g.drop_vs_step1.median():+.4f} · '
              f'median margin {g.margin.median():+.4f}')
    sweep.to_csv(ARCHIVE / 'sweep_pg_step_size.csv', index=False)

if _cleared:
    print(f'\nverdict: USE pg_step_size = {_cleared}. Set it in cell 4, then run cell 7 (all four '
          f'arms).\n         That four-arm table is the M3 result.')
else:
    print("\nverdict: NO L2 SETTING CLEARED. Set attack_type='linf' in cell 4 and re-run this cell."
          "\n         If linf also fails, that negative IS the M3 finding — write it up, do not "
          "keep sweeping.")

### Before `linf` — is the perturbation saturating at the L2 ball?

The sweep says step size changes the perturbation (within-image spread **0.059**, ~17x the 0.0035
same-setting drift) but never changes the verdict: **1 of 10 at every setting, always the same
image.** Two very different explanations fit that, and they point opposite ways:

- **Saturation.** At `step_size >= 2` the optimiser now reaches the `pg_eps = 16` L2 ball and is
  clipped there, so every setting produces the *same-size* perturbation pointing somewhere slightly
  different. Then the binding constraint is **`pg_eps`, not the step size** — and `pg_eps` was only
  ever shown inert **at `step_size = 1`**, where the optimiser never reached the ball. That gate was
  closed under a condition this sweep has just removed.
- **Magnitude is irrelevant.** The perturbation grows with step size and `ssim_adv` still does not
  fall. Then a bigger L2 budget buys nothing and the L2 branch is the wrong geometry — go to `linf`.

The cell below tells them apart by measuring the perturbation directly from the archived PNGs.
**Zero GPU, a few seconds.**


In [ ]:
# 7d · Diagnostic — how big is the shield, actually? Reads archived PNGs only. No GPU.
import numpy as _np
from PIL import Image as _I

def _delta(prot_p, clean_p):
    a = _np.asarray(_I.open(prot_p ).convert('RGB'), _np.float32)
    b = _np.asarray(_I.open(clean_p).convert('RGB').resize(a.shape[1::-1]), _np.float32)
    d = a - b
    return float(_np.sqrt((d ** 2).sum())), float(_np.abs(d).max()), float(_np.abs(d).mean())

_tags = [('step1', 'N_no_wash')] + [(f'step{s}', f'N_step{s}') for s in (2, 4, 8)]
_rows = []
for lab, tag in _tags:
    d = ARCHIVE / tag
    if not (d / 'protected').is_dir():
        print(f'   {tag}: no protected/, skipped'); continue
    for n in sorted(os.listdir(d / 'protected')):
        c = d / 'clean512' / n
        if not c.exists(): continue
        l2, linf, l1 = _delta(d / 'protected' / n, c)
        _rows.append(dict(setting=lab, image=n, L2=round(l2, 1),
                          Linf_levels=round(linf, 1), mean_abs_levels=round(l1, 3)))

dlt = _pd.DataFrame(_rows)
if dlt.empty:
    print('!! nothing measured — are the N_step* archives in this ARCHIVE folder?')
else:
    g = dlt.groupby('setting', sort=False).agg(L2_median=('L2', 'median'),
                                               Linf_median=('Linf_levels', 'median'),
                                               mean_abs=('mean_abs_levels', 'median'))
    display(g)
    dlt.to_csv(ARCHIVE / 'perturbation_size.csv', index=False)

    v = g.L2_median.values
    if len(v) >= 3:
        spread = (v[1:].max() - v[1:].min()) / v[1:].mean() * 100   # across step 2/4/8
        print(f'\nL2 across step 2/4/8 varies by {spread:.1f}% of its mean.')
        if spread < 5:
            print('=> SATURATED. The optimiser is hitting the pg_eps ball and being clipped, so every\n'
                  '   setting spends the same budget in a different direction. pg_eps is the binding\n'
                  '   constraint and it is NO LONGER the inert knob measured at step_size=1.\n'
                  '   NEXT: pg_eps 32 and 64 at the best step size. Do NOT go to linf yet.')
        else:
            print('=> NOT saturated: the shield grows with step size and ssim_adv still does not fall.\n'
                  '   A bigger L2 budget buys nothing. NEXT: attack_type="linf".')
    print(f"\n(for scale: the M2 fidelity finding measured PhotoGuard at ~1 grey level mean, "
          f"our purifier at ~5)")

### The one probe that closes `pg_eps` for good

Cell 7d says the L2 norm **converges**: step 4 -> step 8 moves it by **+0.4% median**. But each
image converges to *its own* ceiling, and those ceilings span **2.8x** (2386 to 6565). A single
global `pg_eps` ball would clip every image to the **same** L2. It does not. So the ball is
probably not what stopped the optimiser — it converged on its own.

"Probably" is not good enough to put in a presentation, and *"did you try a bigger budget?"* is the
first question anyone will ask. One run settles it: **`pg_eps = 64` at `pg_step_size = 4`.**

- L2 stays at each image's step-4 ceiling -> the budget was never binding. **`pg_eps` is closed for
  good, at any step size.** Go to `linf`.
- L2 rises -> the ball *was* binding and the 9 Sep inert-`pg_eps` finding was conditional after all.
  Sweep `pg_eps` instead.

Costs the same as one sweep setting. Run it before `linf`, not after.


In [ ]:
# 7e · ONE probe: does a 4x bigger L2 budget buy a bigger perturbation? (~one sweep setting)
assert not SMOKE and GPU_OK
assert 'dlt' in globals(), 'run cell 7d first — this cell compares against its eps16/step4 L2'
_EPS, _STEP = 64, 4      # <-- the ONLY line to edit. 64 done 19 Sep; set 256 for the last run.

P['pg_eps'], P['pg_step_size'] = _EPS, _STEP
PG = (f"--attack_type={P['attack_type']} --pg_eps={P['pg_eps']} --pg_step_size={P['pg_step_size']} "
      f"--pg_eta={P['pg_eta']} --parallel_index=-1 --device={DEVICE}")   # PG does not follow P

arm = dict(name=f'N_eps{_EPS}_step{_STEP}', pur_iters=None, masked=False)
run_arm(arm)

d, out = ARCHIVE / arm['name'], []
for n in sorted(os.listdir(d / 'protected')):
    if not (d / 'clean512' / n).exists(): continue
    l2, linf, _ = _delta(d / 'protected' / n, d / 'clean512' / n)
    a  = _ssim_pair(d / 'edit_protected' / n, d / 'edit_clean' / n)
    ref = dlt[(dlt.setting == 'step4') & (dlt.image == n)]
    b   = float(ref.L2.iloc[0]) if len(ref) else float('nan')
    out.append(dict(image=n, L2_ref_eps16=round(b, 1), L2_probe=round(l2, 1),
                    growth_pct=round((l2 - b) / b * 100, 1), Linf=round(linf, 1),
                    ssim_adv=round(a, 4), floor=SEED_FLOOR.get(n, float('nan')),
                    engaged=bool(a < SEED_FLOOR.get(n, float('nan')))))

probe = _pd.DataFrame(out); display(probe)
probe.to_csv(ARCHIVE / f'probe_eps{_EPS}_step{_STEP}.csv', index=False)   # per-setting filename

_g, _e = probe.growth_pct.median(), int(probe.engaged.sum())
print(f'\nL2 median growth eps16 -> eps{_EPS} (both at step {_STEP}): {_g:+.1f}%   ·   engaged {_e}/10')
if _g < 5:
    print('=> The budget was NEVER binding. pg_eps is closed at any step size — the optimiser\n'
          '   converges well inside the ball. NEXT: attack_type="linf". Do not sweep pg_eps.')
else:
    print('=> The ball WAS binding. The 9 Sep inert-pg_eps result was conditional on step_size=1.\n'
          '   NEXT: sweep pg_eps (32, 64, 128) at this step size before touching linf.')

### The missing axis — shield fidelity, from archives you already have

Every setting so far has been judged on **engagement only**. That is exactly the one-axis mistake
the Tyro Wash Test was designed to forbid: we hold challengers to **LPIPS <= 0.10** for their
shields, and we have never measured our own test shield's cost at all.

It costs nothing to fix. The `protected/` and `clean512/` PNGs for every setting are already
archived. **Zero GPU.**

This matters now because `pg_eps = 64` is 4x the reference budget and its L-inf reached 96 levels
median. If the shield only becomes measurable once it becomes *visible*, that is the finding — but
it has to be measured, not asserted.


In [ ]:
# 7f · Shield fidelity across every setting run so far. Reads archives only. No GPU.
import lpips as _lpips, torch as _t
_net = _lpips.LPIPS(net='alex').to(DEVICE).eval()

def _lp(a, b):
    f = lambda q: (_t.from_numpy(np.asarray(Image.open(q).convert('RGB'), np.float32) / 127.5 - 1)
                   .permute(2, 0, 1)[None].to(DEVICE))
    with _t.no_grad(): return float(_net(f(a), f(b)))

_settings = [('eps16_step1', 'N_no_wash'), ('eps16_step2', 'N_step2'), ('eps16_step4', 'N_step4'),
             ('eps16_step8', 'N_step8'), ('eps64_step4', 'N_eps64_step4'),
             ('eps256_step4', 'N_eps256_step4')]
_r = []
for lab, tag in _settings:
    d = ARCHIVE / tag
    if not (d / 'protected').is_dir(): continue
    for n in sorted(os.listdir(d / 'protected')):
        if (d / 'clean512' / n).exists():
            _r.append(dict(setting=lab, image=n,
                           shield_lpips=round(_lp(d / 'protected' / n, d / 'clean512' / n), 4)))

fid = _pd.DataFrame(_r)
if fid.empty:
    print('!! no archives found')
else:
    s = fid.groupby('setting', sort=False).shield_lpips.agg(['median', 'max'])
    s['budget_LPIPS_0.10'] = ['OK' if v <= 0.10 else 'OVER BUDGET' for v in s['median']]
    display(s)
    fid.to_csv(ARCHIVE / 'shield_fidelity.csv', index=False)
    print('\nOur published challenger budget is LPIPS <= 0.10. A setting that breaks it is not a\n'
          'shield any defender would deploy, so a wash measured against it proves nothing.')
    print(f"(reference: PhotoGuard at the paper's own default measured ~0.017 in M2)")